# AI-Powered Price Prediction — Training Pipeline

Enterprise ML pipeline for product price suggestion using **Random Forest Regressor**.

**Outputs:** `model.pkl` (production artifact)

**Features:** Category, Brand_Tier, Brand, Original_Price, Age_In_Years, Condition_Score

In [ ]:
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.style.use("dark_background")
sns.set_theme(style="darkgrid", palette="viridis")
print("Libraries loaded.")

## 1. Synthetic Data Generation

We generate 8,000 realistic product records with category-specific depreciation, brand tier multipliers, and condition-based wear factors.

In [ ]:
from train_model import (
    BRANDS,
    CATEGORIES,
    BRAND_TIERS,
    brand_original_price,
    generate_synthetic_data,
)

df = generate_synthetic_data(8000)
df.head(10)

## 2. Exploratory Data Analysis

In [ ]:
print(f"Dataset shape: {df.shape}")
print(df.describe())
print("\nCategory distribution:")
print(df["Category"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(data=df, x="Category", y="Predicted_Price", ax=axes[0])
axes[0].set_title("Price by Category")
axes[0].tick_params(axis="x", rotation=30)

sns.scatterplot(data=df.sample(500), x="Age_In_Years", y="Predicted_Price", hue="Category", alpha=0.6, ax=axes[1])
axes[1].set_title("Depreciation vs Age")

sns.scatterplot(data=df.sample(500), x="Condition_Score", y="Predicted_Price", hue="Brand_Tier", alpha=0.6, ax=axes[2])
axes[2].set_title("Price vs Condition")

plt.tight_layout()
plt.show()

## 3. Model Training — Random Forest Regressor

Pipeline: One-hot encoding for categoricals + standard scaling for numerics → Random Forest (200 trees).

In [ ]:
feature_cols = [
    "Category",
    "Brand_Tier",
    "Brand",
    "Original_Price",
    "Age_In_Years",
    "Condition_Score",
]

X = df[feature_cols]
y = df["Predicted_Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

categorical = ["Category", "Brand_Tier", "Brand"]
numeric = ["Original_Price", "Age_In_Years", "Condition_Score"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("num", StandardScaler(), numeric),
    ]
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        max_depth=18,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

pipeline.fit(X_train, y_train)
print("Training complete.")

## 4. Model Evaluation

In [ ]:
y_pred = pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score:  {r2:.4f}")
print(f"MAE:       ₹{mae:,.2f}")
print(f"RMSE:      ₹{rmse:,.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.4, color="#00F5FF", edgecolors="none")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
axes[0].set_xlabel("Actual Price (₹)")
axes[0].set_ylabel("Predicted Price (₹)")
axes[0].set_title("Actual vs Predicted")

residuals = y_test - y_pred
axes[1].hist(residuals, bins=40, color="#39FF14", edgecolor="black", alpha=0.8)
axes[1].set_xlabel("Residual (₹)")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()

## 5. Confidence Score (Prediction Variance)

Confidence is computed from the standard deviation of individual tree predictions. Lower variance → higher confidence (55–98.5%).

In [ ]:
from train_model import confidence_from_forest

sample = X_test.iloc[[0]]
pred_price = pipeline.predict(sample)[0]
conf = confidence_from_forest(pipeline, sample)

print("Sample prediction:")
print(sample.to_dict("records")[0])
print(f"\nPredicted Price: ₹{pred_price:,.2f}")
print(f"Confidence Score: {conf}%")

## 6. Export Model Artifact

Saves `model.pkl` containing the sklearn pipeline, brand price lookup, and metadata for the Flask API.

In [ ]:
from train_model import train_and_export

artifact = train_and_export("model.pkl")

print(f"Exported model.pkl | Test R²: {artifact['metrics']['r2_test']}")
print(f"Categories: {artifact['categories']}")
print(f"Brands per tier: {list(artifact['brands']['Electronics'].keys())}")